# MCMC Sampling: Metropolis-Hastings and Simulated Annealing

This notebook implements two classic Markov chain Monte Carlo (MCMC) methods and applies them to a bimodal 2D probability distribution:

1. **Metropolis-Hastings (MH)** — a general-purpose MCMC sampler.
2. **Simulated Annealing (SA)** — a variant of MH with a temperature schedule that gradually concentrates on high-probability regions, useful for optimization on top of sampling.

The target distribution is a mixture of two Gaussians with unequal weights and partially overlapping support, which is a standard stress test for MCMC: the chain has to be able to traverse the low-probability region between modes.

## 1. Setup and the target distribution

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import multivariate_normal

EPS = 1.0e-7
np.random.seed(0)

In [ ]:
# Mixture of two 2D Gaussians with unequal weights
mv1 = multivariate_normal(mean=[2.0, 2.0], cov=[[1.0, 0.5], [0.5, 1.0]])
mv2 = multivariate_normal(mean=[-3.0, -3.0], cov=[[1.0, 0.0], [0.0, 1.0]])


def prob(x):
    """Target density: 0.25 * N([2, 2], ...) + 0.75 * N([-3, -3], ...)."""
    return 0.25 * mv1.pdf(x) + 0.75 * mv2.pdf(x)

In [ ]:
def calculate_p(x1, x2):
    """Evaluate the target density on a grid for visualization."""
    p_x = []
    for i in range(len(x1)):
        for j in range(len(x2)):
            p_x.append(prob(np.asarray([[x1[i], x2[j]]])))
    return np.asarray(p_x).reshape(len(x1), len(x2))


x1 = np.linspace(-10.0, 10.0, 400)
x2 = np.linspace(-10.0, 10.0, 400)
p_x = calculate_p(x1, x2).reshape(len(x1), len(x2))

In [ ]:
plt.figure(figsize=(6, 5))
plt.contourf(x1, x2, p_x, 100, cmap='hot')
plt.colorbar(label='p(x)')
plt.title('Target distribution (bimodal Gaussian mixture)')
plt.xlabel('x1')
plt.ylabel('x2')
plt.show()

The distribution has **two modes**, located near `(2, 2)` and `(-3, -3)`. Because the second component has weight 0.75, the most probable point is around `(-3, -3)`.

## 2. Metropolis-Hastings

The MH algorithm draws samples from a target density `p(x)` using a proposal distribution `q(x_new | x_old)`. At each step:

1. Propose a candidate `x_new ~ q(x_new | x_old)`.
2. Compute the acceptance ratio `A = min(1, p(x_new)/p(x_old))` (assuming a symmetric proposal).
3. Accept with probability `A`, otherwise stay at `x_old`.

We use a Gaussian random-walk proposal centered at the current point, with controllable standard deviation. Because the Gaussian proposal has full support and is symmetric, the resulting Markov chain is irreducible and aperiodic — both required for the chain to converge to the target distribution.

In [ ]:
class MetropolisHastings:
    """Metropolis-Hastings sampler with a Gaussian random-walk proposal."""

    def __init__(self, x, prob, std=0.1):
        self.prob = prob
        self.std = std
        self.x_old = x

    def proposal(self, x):
        # Symmetric Gaussian random-walk proposal
        return np.random.normal(x, self.std)

    def evaluate(self, x_new, x_old):
        p_old = self.prob(x_old)
        p_new = self.prob(x_new)
        A = p_new / (p_old + EPS)
        return np.minimum(1.0, A)

    def select(self, x_new, A):
        u = np.random.uniform()
        if u < A:
            self.x_old = x_new
        return self.x_old

    def step(self):
        x_prop = self.proposal(self.x_old)
        A = self.evaluate(x_prop, self.x_old)
        return self.select(x_prop, A)

In [ ]:
def plot_sampling_process(ax, sampler, title, num_epochs):
    """Run a sampler for `num_epochs` steps and overlay accepted samples on the target density."""
    ax.contourf(x1, x2, p_x / p_x.sum(), 100, cmap='hot')

    x_samp = sampler.x_old
    count = 0
    for _ in range(num_epochs):
        x = sampler.step()
        if (x == x_samp[-1]).all():
            continue
        count += 1
        x_samp = np.concatenate((x_samp, x), 0)

    ax.scatter(x_samp[:, 0], x_samp[:, 1], marker='+')
    ax.set_title(f'{title} AR={count / num_epochs:.2f}')

In [ ]:
# Run MH with two different proposal scales, three repeats each
num_epochs = 1500
stds = [0.1, 0.1, 0.1, 0.5, 0.5, 0.5]

fig, axs = plt.subplots(1, len(stds), figsize=(20, 3))
fig.suptitle('Metropolis-Hastings — samples for different proposal stds (3 repeats each)', y=1.05)
fig.tight_layout()

x_init = np.asarray([[0.0, 0.0]])

for i, std in enumerate(stds):
    mh = MetropolisHastings(x=x_init, prob=prob, std=std)
    plot_sampling_process(axs[i], sampler=mh, title=f'std={std}', num_epochs=num_epochs)

plt.show()

### Observations on Metropolis-Hastings

- With `std = 0.1` the chain mixes locally and rarely escapes the mode it started exploring. Even though it is irreducible in theory, in practice 1500 iterations are not enough to reliably hop between the two modes.
- With `std = 0.5` the chain can jump between modes, but the **acceptance ratio (AR)** drops because larger steps frequently land in low-density regions and get rejected.

This is the classic **mixing vs. acceptance** trade-off in random-walk MH: small steps are accepted often but mix slowly; large steps mix faster but waste evaluations on rejected proposals.

## 3. Simulated Annealing

Simulated Annealing is structurally similar to MH but operates on a **tempered** density `p(x)^(1/T)` and gradually cools the temperature `T` toward zero. As `T -> 0` the tempered density concentrates on the global mode, so SA naturally drifts toward high-probability regions and is therefore more useful for **optimization** than for representative sampling.

We use the logarithmic cooling schedule:

$$
T_t = \frac{1}{C \cdot \log(t + T_0)}
$$

which is a standard convergence-guaranteeing choice.

**Difference from MH:** MH targets the stationary distribution `p(x)`. SA's stationary distribution shrinks toward the maximum of `p(x)` as the temperature drops, so SA is preferred when you want the mode rather than samples from the whole distribution.

In [ ]:
class SimulatedAnnealing:
    """Simulated annealing with a logarithmic cooling schedule."""

    def __init__(self, x, prob, std=0.1, T0=1.0, C=1.0):
        self.prob = prob
        self.std = std
        self.x_old = x
        self.T0 = T0
        self.C = C
        self.t = 1  # start at 1 so the cooling schedule never evaluates log(0) when T0=1

    def proposal(self, x):
        return np.random.normal(x, self.std)

    def evaluate(self, x_new, x_old, T):
        # Tempered ratio: p(x_new)^(1/T) / p(x_old)^(1/T)
        p_old = self.prob(x_old) ** (1 / T)
        p_new = self.prob(x_new) ** (1 / T)
        A = p_new / (p_old + EPS)
        return np.minimum(1.0, A)

    def select(self, x_new, A):
        u = np.random.uniform()
        if u < A:
            self.x_old = x_new
        return self.x_old

    def step(self):
        # Logarithmic cooling schedule
        T = (self.C * np.log(self.t + self.T0)) ** -1
        self.t += 1
        x_prop = self.proposal(self.x_old)
        A = self.evaluate(x_prop, self.x_old, T)
        return self.select(x_prop, A)

In [ ]:
# Sweep over std, T0, and C
num_epochs = 1500
stds_sa = [0.1, 0.1, 0.1, 0.5, 0.5, 0.5]
T0s = [0.1, 1.0, 10.0]
Cs = [0.1, 1.0, 10.0]

fig, axs = plt.subplots(len(Cs) * len(T0s), len(stds_sa), figsize=(20, 18))
fig.suptitle('Simulated Annealing — sweep over std (cols) and (T0, C) (rows)', y=1.005)
fig.tight_layout()

x_init = np.asarray([[0.0, 0.0]])

for i, C in enumerate(Cs):
    for k, T0 in enumerate(T0s):
        for j, std in enumerate(stds_sa):
            sa = SimulatedAnnealing(x=x_init, prob=prob, std=std, T0=T0, C=C)
            plot_sampling_process(
                axs[len(T0s) * i + k, j],
                sampler=sa,
                title=f'std={sa.std} T0={T0} C={C}',
                num_epochs=num_epochs,
            )

plt.show()

### Observations on Simulated Annealing

- **`std` (proposal scale).** Small `std` produces tight, mode-local clusters; large `std` allows mode hopping but lowers acceptance.
- **`T0` (initial temperature).** A low `T0` makes SA behave almost greedily from the start; a high `T0` allows broad exploration before the cooling kicks in.
- **`C` (cooling rate).** Large `C` cools the chain quickly and locks onto whatever mode it found early. Small `C` cools slowly, giving the chain more time to explore.

Best sampling-quality setting in this sweep: roughly `std=0.5`, `T0=1.0`, `C=0.1` — large enough proposals to traverse modes, plus slow cooling so the chain doesn't commit prematurely.

## 4. Comparison: MH vs SA

| Aspect              | Metropolis-Hastings                   | Simulated Annealing                       |
|---------------------|----------------------------------------|--------------------------------------------|
| Target              | The full distribution `p(x)`           | The mode of `p(x)` (as T → 0)              |
| Hyperparameters     | Proposal std                           | Proposal std + `T0` + cooling rate `C`     |
| Use case            | Posterior sampling, expectations       | Global optimization                        |
| Difficulty          | Easier to tune, fewer knobs            | More knobs, but more flexible              |

**Which performed better?**
For *sampling* the bimodal target, MH with a moderate `std=0.5` does a reasonable job once it manages to cross between modes. SA, when configured for slow cooling, can also represent both modes, but it is fundamentally biased toward the dominant mode at `(-3, -3)` and so is closer to a mode-finder than a faithful sampler.

**Which is easier to use?**
MH wins on simplicity — only the proposal needs to be chosen. SA is more flexible (it doubles as an optimizer) but introduces extra hyperparameters that interact non-trivially.